In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

print("환경 연결 성공")

환경 연결 성공


In [2]:
clinical_path = "../data/raw/clinical/TCGA-CDR-SupplementalTableS1.xlsx"

df = pd.read_excel(clinical_path)

print("전체 데이터:", df.shape)
df.head()

전체 데이터: (11160, 34)


,Unnamed: 0,bcr_patient_barcode,type,age_at_initial_pathologic_diagnosis,gender,race,ajcc_pathologic_tumor_stage,clinical_stage,histological_type,histological_grade,...,residual_tumor,OS,OS.time,DSS,DSS.time,DFI,DFI.time,PFI,PFI.time,Redaction
0,1,TCGA-OR-A5J1,ACC,58.0,MALE,WHITE,Stage II,[Not Applicable],Adrenocortical carcinoma- Usual Type,[Not Available],...,NaN,1.0,1355.0,1.0,1355.0,1.0,754.0,1.0,754.0,NaN
1,2,TCGA-OR-A5J2,ACC,44.0,FEMALE,WHITE,Stage IV,[Not Applicable],Adrenocortical carcinoma- Usual Type,[Not Available],...,NaN,1.0,1677.0,1.0,1677.0,NaN,NaN,1.0,289.0,NaN
2,3,TCGA-OR-A5J3,ACC,23.0,FEMALE,WHITE,Stage III,[Not Applicable],Adrenocortical carcinoma- Usual Type,[Not Available],...,NaN,0.0,2091.0,0.0,2091.0,1.0,53.0,1.0,53.0,NaN
3,4,TCGA-OR-A5J4,ACC,23.0,FEMALE,WHITE,Stage IV,[Not Applicable],Adrenocortical carcinoma- Usual Type,[Not Available],...,NaN,1.0,423.0,1.0,423.0,NaN,NaN,1.0,126.0,NaN
4,5,TCGA-OR-A5J5,ACC,30.0,MALE,WHITE,Stage III,[Not Applicable],Adrenocortical carcinoma- Usual Type,[Not Available],...,NaN,1.0,365.0,1.0,365.0,NaN,NaN,1.0,50.0,NaN


In [3]:
kirc = df[df["type"] == "KIRC"].copy()

print("KIRC 데이터:", kirc.shape)
kirc.head()

KIRC 데이터: (537, 34)


,Unnamed: 0,bcr_patient_barcode,type,age_at_initial_pathologic_diagnosis,gender,race,ajcc_pathologic_tumor_stage,clinical_stage,histological_type,histological_grade,...,residual_tumor,OS,OS.time,DSS,DSS.time,DFI,DFI.time,PFI,PFI.time,Redaction
3882,3883,TCGA-3Z-A93Z,KIRC,69.0,MALE,BLACK OR AFRICAN AMERICAN,Stage I,[Not Applicable],Kidney Clear Cell Renal Carcinoma,G2,...,NaN,0.0,385.0,0.0,385.0,0.0,385.0,0.0,385.0,NaN
3883,3884,TCGA-6D-AA2E,KIRC,68.0,FEMALE,BLACK OR AFRICAN AMERICAN,Stage I,[Not Applicable],Kidney Clear Cell Renal Carcinoma,G2,...,NaN,0.0,362.0,0.0,362.0,NaN,NaN,0.0,362.0,NaN
3884,3885,TCGA-A3-3306,KIRC,67.0,MALE,WHITE,Stage I,[Not Applicable],Kidney Clear Cell Renal Carcinoma,G3,...,NaN,0.0,1120.0,0.0,1120.0,NaN,NaN,0.0,1120.0,NaN
3885,3886,TCGA-A3-3307,KIRC,66.0,MALE,[Not Available],Stage III,[Not Applicable],Kidney Clear Cell Renal Carcinoma,G3,...,NaN,0.0,1436.0,0.0,1436.0,NaN,NaN,0.0,1436.0,NaN
3886,3887,TCGA-A3-3308,KIRC,77.0,FEMALE,WHITE,Stage III,[Not Applicable],Kidney Clear Cell Renal Carcinoma,G2,...,NaN,0.0,16.0,0.0,16.0,NaN,NaN,0.0,16.0,NaN


In [4]:
print("전체 KIRC 행 수:", len(kirc))
print("고유 환자 수:", kirc["bcr_patient_barcode"].nunique())
print("중복 환자 수:", kirc["bcr_patient_barcode"].duplicated().sum())

print("\nPFI 결측:", kirc["PFI"].isna().sum())
print("PFI.time 결측:", kirc["PFI.time"].isna().sum())

print("\nPFI 분포")
print(kirc["PFI"].value_counts(dropna=False))

print("\nPFI event 비율")
print(kirc["PFI"].mean())

전체 KIRC 행 수: 537
고유 환자 수: 537
중복 환자 수: 0

PFI 결측: 0
PFI.time 결측: 2

PFI 분포
PFI
0.0    375
1.0    162
Name: count, dtype: int64

PFI event 비율
0.3016759776536313


In [21]:
print("PFI.time 분포")
print(kirc["PFI.time"].describe())

print("\n1년 이상 추적:", (kirc["PFI.time"] >= 365).sum())
print("3년 이상 추적:", (kirc["PFI.time"] >= 365 * 3).sum())
print("5년 이상 추적:", (kirc["PFI.time"] >= 365 * 5).sum())

PFI.time 분포
count     535.000000
mean     1160.712150
std       946.066312
min         0.000000
25%       360.500000
50%      1011.000000
75%      1756.000000
max      4074.000000
Name: PFI.time, dtype: float64

1년 이상 추적: 400
3년 이상 추적: 255
5년 이상 추적: 123


In [22]:
clinical = kirc[
    [
        "bcr_patient_barcode",
        "age_at_initial_pathologic_diagnosis",
        "gender",
        "ajcc_pathologic_tumor_stage",
        "histological_grade",
        "PFI",
        "PFI.time"
    ]
].copy()

clinical = clinical.dropna(subset=["PFI", "PFI.time"])

clinical = clinical.rename(columns={
    "bcr_patient_barcode": "patient_id",
    "age_at_initial_pathologic_diagnosis": "age",
    "ajcc_pathologic_tumor_stage": "stage",
    "histological_grade": "grade",
    "PFI": "pfi_event",
    "PFI.time": "pfi_time"
})

print(clinical.shape)
clinical.head()

(535, 7)


,patient_id,age,gender,stage,grade,pfi_event,pfi_time
3882,TCGA-3Z-A93Z,69.0,MALE,Stage I,G2,0.0,385.0
3883,TCGA-6D-AA2E,68.0,FEMALE,Stage I,G2,0.0,362.0
3884,TCGA-A3-3306,67.0,MALE,Stage I,G3,0.0,1120.0
3885,TCGA-A3-3307,66.0,MALE,Stage III,G3,0.0,1436.0
3886,TCGA-A3-3308,77.0,FEMALE,Stage III,G2,0.0,16.0


In [23]:
clinical["stage"] = clinical["stage"].replace({
    "[Discrepancy]": "Unknown"
})

clinical["grade"] = clinical["grade"].replace({
    "GX": "Unknown",
    "[Not Available]": "Unknown"
})

print(clinical["stage"].value_counts(dropna=False))
print()
print(clinical["grade"].value_counts(dropna=False))

stage
Stage I      269
Stage III    125
Stage IV      81
Stage II      57
Unknown        3
Name: count, dtype: int64

grade
G2         230
G3         207
G4          76
G1          14
Unknown      8
Name: count, dtype: int64


In [24]:
from pathlib import Path

output_path = Path("../data/interim/kirc_clinical_pfi.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

clinical.to_parquet(output_path, index=False)

print("저장 완료:", output_path)

저장 완료: ..\data\interim\kirc_clinical_pfi.parquet


In [25]:
check = pd.read_parquet("../data/interim/kirc_clinical_pfi.parquet")

print(check.shape)
print(check.columns.tolist())
check.head()

(535, 7)
['patient_id', 'age', 'gender', 'stage', 'grade', 'pfi_event', 'pfi_time']


,patient_id,age,gender,stage,grade,pfi_event,pfi_time
0,TCGA-3Z-A93Z,69.0,MALE,Stage I,G2,0.0,385.0
1,TCGA-6D-AA2E,68.0,FEMALE,Stage I,G2,0.0,362.0
2,TCGA-A3-3306,67.0,MALE,Stage I,G3,0.0,1120.0
3,TCGA-A3-3307,66.0,MALE,Stage III,G3,0.0,1436.0
4,TCGA-A3-3308,77.0,FEMALE,Stage III,G2,0.0,16.0
